# 4.2 VAE Imputation: Inferenz & Anwendung

<div class="alert alert-info">

## Ziel
Anwendung der trainierten VAE-Modelle aus 4.1.
Hier wird die "Masking"-Strategie angewandt: Ein Feature wird aus den **vollständigen Daten** künstlich entfernt, und der VAE versucht, es zu rekonstruieren.

## Vorgehen
1. **Modelle laden**: Den neuesten Run-Ordner aus 4.1 identifizieren.
2. **Iteration**: Für jedes trainierte Modell (Run_001 bis Run_030):
    - Laden von Modell, Scaler und Metadaten.
    - Laden der entsprechenden vollständigen Daten.
3. **Masking & Imputation**:
    - Für jedes Feature wird eine Kopie des Datensatzes erstellt.
    - Das Feature wird auf 0 (oder Mean) gesetzt (maskiert).
    - Das Modell rekonstruiert den Input.
    - Der rekonstruierte Wert für das maskierte Feature wird gespeichert.
4. **Output**: Eine CSV-Datei pro Run mit `Original` und `Imputed` Werten für die Evaluation in 4.3.
</div>

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import joblib
import json
from pathlib import Path
from datetime import datetime

# Reproduzierbarkeit
torch.manual_seed(42)
np.random.seed(42)

In [2]:
# ------------------------- VAE Model Definition (High Capacity) -------------------------
class VAE(nn.Module):
    def __init__(self, input_dim, latent_dim=16, hidden_dim=512):
        super(VAE, self).__init__()
        
        # ------------------------- Encoder -------------------------
        self.enc1 = nn.Linear(input_dim, hidden_dim)
        self.enc2 = nn.Linear(hidden_dim, hidden_dim // 2)
        
        self.fc_mu = nn.Linear(hidden_dim // 2, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim // 2, latent_dim)
        
        # ------------------------- Decoder -------------------------
        self.dec1 = nn.Linear(latent_dim, hidden_dim // 2)
        self.dec2 = nn.Linear(hidden_dim // 2, hidden_dim)
        self.dec3 = nn.Linear(hidden_dim, input_dim)
        
        self.leaky_relu = nn.LeakyReLU(0.2)

    def encode(self, x):
        x = self.leaky_relu(self.enc1(x))
        x = self.leaky_relu(self.enc2(x))
        return self.fc_mu(x), self.fc_logvar(x)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5*logvar)
        eps = torch.randn_like(std)
        return mu + eps*std

    def decode(self, z):
        z = self.leaky_relu(self.dec1(z))
        z = self.leaky_relu(self.dec2(z))
        return self.dec3(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar


In [ ]:
# ------------------------- Pfade finden -------------------------
base_dir = Path.cwd()
models_root = base_dir.parent / "4.1_VAE_Imputation" / "Models"

# Neuesten Modell-Ordner suchen
timestamp_folders = [f for f in models_root.iterdir() if f.is_dir()]
if not timestamp_folders:
    raise FileNotFoundError("Keine trainierten Modelle in 4.1 gefunden!")

latest_model_folder = max(timestamp_folders, key=lambda f: f.stat().st_mtime)
print(f"Nutze Modelle aus: {latest_model_folder.name}")

# ------------------------- Datenquelle (Preprocessing 3.1) -------------------------
preprocessing_root = base_dir.parent.parent / "3_Machine-Learning" / "3.1_Preprocessing" / "Preprocessing"
latest_prep_folder = max([f for f in preprocessing_root.iterdir() if f.is_dir()], key=lambda f: f.stat().st_mtime)
input_path_prep = latest_prep_folder / "Preprocessed_SOM_Ready.csv"
df_full = pd.read_csv(input_path_prep, low_memory=False)
print(f"Daten geladen: {latest_prep_folder.name}")

Nutze Modelle aus: 2026-01-10_01-27-36


Daten geladen: 2026-01-09_14-06-57


In [ ]:
# ------------------------- Inferenz Monitor Loop -------------------------
import time

# ------------------------- Output Ordner Definition  -------------------------
results_dir = base_dir / "Imputation_Results" / latest_model_folder.name
results_dir.mkdir(parents=True, exist_ok=True)
print(f"Saving results to: {results_dir.name}")

processed_runs = set()
print("Starte Monitoring auf neue Modelle...")

while True:
    # ------------------------- Alle Meta-Daten finden -------------------------
    meta_files = sorted(list(latest_model_folder.glob("*_meta.json")))
    
    # ------------------------- Filtern auf neue -------------------------
    new_files = [f for f in meta_files if f.name not in processed_runs]
    
    if not new_files:
        #  ------------------------- Scghauen obs beendet wurde -------------------------
        if (latest_model_folder / "DONE_TRAINING").exists():
            print("\nDONE_TRAINING erkannt. Keine neuen Modelle mehr. Beende Inferenz.")
            (results_dir / "DONE_INFERENCE").touch()
            print(f"Signal file created: DONE_INFERENCE")
            break
        else:
            print(".", end="", flush=True)
            time.sleep(5)
            continue
            
    for meta_file in new_files:
        processed_runs.add(meta_file.name)
        
        # ------------------------- Metadaten lesen -------------------------
        with open(meta_file, 'r') as f:
            meta = json.load(f)
            
        run_id = meta['run_id']
        features = meta['features_mapped']
        latent_dim = meta['latent_dim'] #-------------------------  Falls dynamisch, hier auslesen
        hidden_dim_meta = meta.get('hidden_dim', 512) # ------------------------- Fallback wenn fehlend
        
        print(f"\nProcessing {run_id}...")
        
        # ------------------------- -------------------------Modell & Scaler laden -------------------------
        model_path = latest_model_folder / f"{run_id}_vae.pth"
        scaler_path = latest_model_folder / f"{run_id}_scaler.joblib"
        
        # ------------------------- Sicherstellen dass alles da ist -------------------------
        wait_count = 0
        while not (model_path.exists() and scaler_path.exists()):
            time.sleep(1)
            wait_count += 1
            if wait_count > 10: break
            
        if not model_path.exists(): continue
        
        scaler = joblib.load(scaler_path)
        input_dim = len(features)
        
        vae = VAE(input_dim, latent_dim=latent_dim, hidden_dim=hidden_dim_meta)
        
        # ------------------------- Fehlerbehandlung, da viele Zugriffsprobleme in Vergangenheit -------------------------
        for attempt in range(5):
            try:
                vae.load_state_dict(torch.load(model_path))
                break
            except PermissionError:
                if attempt < 4: time.sleep(1)
                else: raise
        vae.eval()
        
        # ------------------------- Daten vorbereiten -------------------------
        df_run = df_full[features].dropna().copy()
        
        # ------------------------- Für Evaluation: 5000 Samples -------------------------
        if len(df_run) > 5000: df_run = df_run.sample(5000, random_state=42)
        
        original_values = scaler.transform(df_run.values) 
        original_data_scaled = scaler.transform(df_run.values)
        
        # ------------------------- Spalten nullen -------------------------
        results_list = []
        
        for i, feature_name in enumerate(features):
            masked_data = original_data_scaled.copy()
            masked_data[:, i] = 0.0 
            
            tensor_in = torch.from_numpy(masked_data).float()
            with torch.no_grad():
                recon, _, _ = vae(tensor_in)
                
            recon_numpy = recon.numpy()

            imputed_column_scaled = recon_numpy[:, i]
            original_column_scaled = original_data_scaled[:, i]
            
            dummy_matrix = np.zeros_like(original_data_scaled)
            dummy_matrix[:, i] = imputed_column_scaled
            imputed_vals_real = scaler.inverse_transform(dummy_matrix)[:, i]
            
            dummy_matrix[:, i] = original_column_scaled
            original_vals_real = scaler.inverse_transform(dummy_matrix)[:, i]
            
            temp_df = pd.DataFrame({
                'Feature': feature_name,
                'Original': original_vals_real,
                'Imputed': imputed_vals_real,
                'Run_ID': run_id
            })
            results_list.append(temp_df)
            
        final_df_run = pd.concat(results_list, axis=0)
        out_file = results_dir / f"Imputation_Results_{run_id}.csv"
        final_df_run.to_csv(out_file, index=False)
        print(f"Saved: {out_file.name}")
        
print("\nInferenz Loop Abgeschlossen")


Saving results to: 2026-01-10_01-27-36
Starte Monitoring auf neue Modelle...
.

.

.

.

.

...

Saved: Imputation_Results_Run_027.csv
.


Processing Run_029...
Saved: Imputation_Results_Run_029.csv
.

.

.


Processing Run_030...
Saved: Imputation_Results_Run_030.csv

DONE_TRAINING erkannt. Keine neuen Modelle mehr. Beende Inferenz.
Signal file created: DONE_INFERENCE

--- Inferenz Loop Abgeschlossen ---
